# Spindle inspection — EL3023 (YA)

Runs `yasa.spindles_detect` on E90 / E9 / E144 over clean N2 only, then builds one
single-channel `Raw` per channel with the detected spindles as annotations (`SP<idx>`)
so they can be browsed.

Order: run cells 1–6, then use `browse('E90')` etc. Write down the times that look good.

In [1]:
%matplotlib qt
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa

REPO = Path.cwd() if Path.cwd().name != 'code' else Path.cwd().parent
sys.path.insert(0, str(REPO / 'code'))
from utils.utils import find_subject_fif_file

SUBJECT = 'EL3023'
GROUP_DIR = Path('I:/Shaked/ISO_data/control_clean')
CHANNELS = ['E90', 'E9', 'E144']

SIGMA_BAND = (11, 16)          # band for the auxiliary filtered trace
INCLUDE_SIGMA_TRACE = True     # adds '<ch>_sigma' next to each channel in the browser

OUT_DIR = REPO / 'results' / 'spindle_examples'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(REPO, OUT_DIR)

i:\Shaked\ISO i:\Shaked\ISO\results\spindle_examples


In [2]:
# 1. Load the avg-ref / interpolated raw, keeping only the three channels
fif_path = find_subject_fif_file(GROUP_DIR / SUBJECT)
raw = mne.io.read_raw_fif(fif_path, preload=False, verbose='error')

bad_txt = (GROUP_DIR / SUBJECT / f'{SUBJECT}_bad_channels.txt').read_text().split()
print('missing from file :', [c for c in CHANNELS if c not in raw.ch_names])
print('flagged as bad    :', [c for c in CHANNELS if c in bad_txt], '(interpolated in this file)')
print('projectors        :', raw.info['projs'])

raw.pick(CHANNELS).load_data()
sfreq = raw.info['sfreq']
print(raw)

Using file: EL3023_176-channels_resample250_filtered_scored_bad-epochs_avgref_interpolate_raw.fif
missing from file : []
flagged as bad    : [] (interpolated in this file)
projectors        : []
Reading 0 ... 7078095  =      0.000 ... 28312.380 secs...
<Raw | EL3023_176-channels_resample250_filtered_scored_bad-epochs_avgref_interpolate_raw.fif, 3 x 7078096 (28312.4 s), ~162.1 MB, data loaded>


In [3]:
# 2. Sample-wise hypnogram for yasa: N2 -> 2, other stages -> their code, BAD_* -> -1
STAGE_CODE = {'Wake': 0, 'WAKE': 0, 'NREM1': 1, 'N1': 1, 'NREM2': 2,
              'N2': 2, 'NREM3': 3, 'N3': 3, 'REM': 4}

n_samples = raw.n_times
hypno = np.full(n_samples, -2, dtype=int)   # -2 = unscored


def _span(ann):
    s = max(int(ann['onset'] * sfreq), 0)
    e = min(int((ann['onset'] + ann['duration']) * sfreq), n_samples)
    return s, e


for ann in raw.annotations:                      # stages first ...
    code = STAGE_CODE.get(str(ann['description']))
    if code is not None:
        s, e = _span(ann)
        hypno[s:e] = code

for ann in raw.annotations:                      # ... then BAD wins over the stage
    if 'BAD' in str(ann['description']).upper():
        s, e = _span(ann)
        hypno[s:e] = -1

print(f'clean N2 available : {(hypno == 2).sum() / sfreq / 60:.1f} min')
print(f'N2 dropped as BAD  : {(hypno == -1).sum() / sfreq / 60:.1f} min (all stages)')

clean N2 available : 175.7 min
N2 dropped as BAD  : 20.7 min (all stages)


In [4]:
# 3. Spindle detection, restricted to clean N2
sp = yasa.spindles_detect(raw, hypno=hypno, include=(2,))
assert sp is not None, 'no spindles detected'

sp_df = sp.summary().reset_index(drop=True)
sp_df.to_csv(OUT_DIR / f'{SUBJECT}_spindles.csv', index=False)
print(f'{len(sp_df)} spindles -> {OUT_DIR / (SUBJECT + "_spindles.csv")}')
sp.summary(grp_chan=True)

[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.3s finished
[Parallel(n_jobs=1)]: Done   3 out of   3 | elapsed:    0.4s finished


2499 spindles -> i:\Shaked\ISO\results\spindle_examples\EL3023_spindles.csv


,Count,Duration,Amplitude,RMS,AbsPower,RelPower,Frequency,Oscillations,Symmetry
Channel,,,,,,,,,
E144,879,0.936369,20.925272,4.489054,1.261024,0.405338,13.762967,12.274175,0.502898
E9,580,0.872414,24.595000,5.301636,1.394179,0.383730,13.663124,11.277586,0.484131
E90,1040,1.019365,23.655853,5.060353,1.389007,0.442709,13.746462,13.473077,0.499133


In [ ]:
# 4. Per-channel Raw objects, spindles added as annotations (kept alongside stages/BADs)
#    Labels are numbered PER CHANNEL (SP1..SP_n within each channel) and the same number
#    is stored in sp_df['sp_num'], so an SP label can always be traced back to its row.
sp_df['sp_num'] = sp_df.groupby('Channel').cumcount() + 1

raws = {}
for ch in CHANNELS:
    r = raw.copy().pick([ch])
    if INCLUDE_SIGMA_TRACE:
        sig = r.copy().filter(*SIGMA_BAND, verbose='error')
        sig.rename_channels({ch: f'{ch}_sigma'})
        r.add_channels([sig], force_update_info=True)

    ch_sp = sp_df[sp_df['Channel'] == ch]
    ann = r.annotations.copy()
    ann.append(ch_sp['Start'].to_numpy(),
               ch_sp['Duration'].to_numpy(),
               [f'SP{n}' for n in ch_sp['sp_num']])
    r.set_annotations(ann)
    raws[ch] = r
    print(f'{ch}: {len(ch_sp)} spindles (SP1-SP{len(ch_sp)}), traces -> {r.ch_names}')

sp_df.to_csv(OUT_DIR / f'{SUBJECT}_spindles.csv', index=False)

In [6]:
# 5. Where to look: 30 s windows containing the most spindles (per channel)
WIN = 30
for ch in CHANNELS:
    ch_sp = sp_df[sp_df['Channel'] == ch].copy()
    ch_sp['win'] = (ch_sp['Start'] // WIN).astype(int)
    top = ch_sp.groupby('win').agg(n=('Start', 'size'),
                                   mean_amp=('Amplitude', 'mean'),
                                   mean_dur=('Duration', 'mean'))
    top = top.sort_values(['n', 'mean_amp'], ascending=False).head(8)
    print(f'\n=== {ch} — densest 30 s windows (t0 in seconds) ===')
    for win, row in top.iterrows():
        print(f'  t0={win * WIN:>7.0f}s  ({win * WIN / 60:6.1f} min)  '
              f"n={int(row['n'])}  amp={row['mean_amp']:.1f}uV  dur={row['mean_dur']:.2f}s")


=== E90 — densest 30 s windows (t0 in seconds) ===
  t0=  10860s  ( 181.0 min)  n=7  amp=23.2uV  dur=0.76s
  t0=  14010s  ( 233.5 min)  n=7  amp=22.5uV  dur=1.25s
  t0=   1230s  (  20.5 min)  n=6  amp=31.8uV  dur=1.62s
  t0=  24090s  ( 401.5 min)  n=6  amp=28.4uV  dur=1.00s
  t0=  28170s  ( 469.5 min)  n=6  amp=27.9uV  dur=1.23s
  t0=  14280s  ( 238.0 min)  n=6  amp=26.9uV  dur=1.16s
  t0=  22830s  ( 380.5 min)  n=6  amp=26.3uV  dur=1.16s
  t0=  27810s  ( 463.5 min)  n=6  amp=25.6uV  dur=1.21s

=== E9 — densest 30 s windows (t0 in seconds) ===
  t0=  24090s  ( 401.5 min)  n=5  amp=32.0uV  dur=0.84s
  t0=  10860s  ( 181.0 min)  n=5  amp=27.5uV  dur=0.84s
  t0=  23280s  ( 388.0 min)  n=5  amp=26.5uV  dur=0.87s
  t0=  22350s  ( 372.5 min)  n=5  amp=26.3uV  dur=0.85s
  t0=  20760s  ( 346.0 min)  n=5  amp=24.8uV  dur=1.28s
  t0=  23010s  ( 383.5 min)  n=5  amp=23.3uV  dur=0.76s
  t0=  10020s  ( 167.0 min)  n=5  amp=18.7uV  dur=0.61s
  t0=   1230s  (  20.5 min)  n=4  amp=28.4uV  dur=1.25s



In [8]:
# 6. Browse. Spindles show up as SP<idx> annotations; use the printed t0 values as `start`.
def browse(ch, start=0.0, duration=30.0, scale_uv=75):
    return raws[ch].plot(start=start, duration=duration,
                         scalings=dict(eeg=scale_uv * 1e-6),
                         title=f'{SUBJECT} — {ch}', show_scrollbars=True,
                         block=False)


browse('E90', start=0)

C:\Users\Shaked\AppData\Local\Temp\6\ipykernel_22084\3450373359.py:3: RuntimeWarning: Requested theme file not found, will use light instead: 'bright'
  return raws[ch].plot(start=start, duration=duration,


In [ ]:
browse('E9', start=0)

In [ ]:
browse('E144', start=0)

## Scratch preview
Quick static look at a candidate window before it becomes a real figure —
broadband on top, 11–16 Hz below, detected spindles shaded.

In [ ]:
def preview(ch, t0, dur=20.0, save=None):
    r = raw.copy().pick([ch])
    sig = r.copy().filter(*SIGMA_BAND, verbose='error')
    t = r.times
    m = (t >= t0) & (t <= t0 + dur)
    broad = r.get_data()[0][m] * 1e6
    filt = sig.get_data()[0][m] * 1e6

    fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
    axes[0].plot(t[m], broad, lw=0.7, color='k')
    axes[0].set_ylabel('EEG (uV)')
    axes[1].plot(t[m], filt, lw=0.7, color='tab:blue')
    axes[1].set_ylabel(f'{SIGMA_BAND[0]}-{SIGMA_BAND[1]} Hz (uV)')
    axes[1].set_xlabel('Time (s)')

    win_sp = sp_df[(sp_df['Channel'] == ch) &
                   (sp_df['Start'] < t0 + dur) & (sp_df['End'] > t0)]
    for _, s in win_sp.iterrows():
        for ax in axes:
            ax.axvspan(s['Start'], s['End'], color='tab:orange', alpha=0.25, lw=0)

    axes[0].set_title(f'{SUBJECT} — {ch} — {t0:.1f}-{t0 + dur:.1f} s '
                      f'({len(win_sp)} detected)')
    for ax in axes:
        ax.set_xlim(t0, t0 + dur)
        ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    if save:
        fig.savefig(OUT_DIR / save, dpi=300, bbox_inches='tight')
        print('saved', OUT_DIR / save)
    return fig


# preview('E90', t0=<seconds>, dur=20)